In [0]:
# ─────────────────────────────────────────────────────────────
# GOLD DIMENSION VIEWS  (presentation layer over silver)
# Views = always in sync with silver, zero extra storage/maintenance.
# Each resolves curated enrichment; PK is the *_key the fact references.
# ─────────────────────────────────────────────────────────────
CATALOG = "cricket"

# --- dim_player: fold in enrichment seed (null until you populate it) ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_player AS
SELECT
    p.person_id,
    p.canonical_name,
    COALESCE(e.full_name, p.canonical_name) AS display_name,
    p.gender,
    e.country,
    e.dob,
    e.batting_style,
    e.bowling_style,
    e.bat_role,
    e.bowl_role,
    e.is_wicketkeeper,
    p.first_match_date,
    p.last_match_date
FROM {CATALOG}.silver.dim_player p
LEFT JOIN {CATALOG}.silver.dim_player_enrichment e USING (person_id)
""")

In [0]:
# --- dim_team: resolve franchise canonical (falls back to raw name) ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_team AS
SELECT
    team_key,
    team_name,
    COALESCE(franchise_current_name, team_name) AS display_team,
    franchise_key,
    team_type,
    first_match_date,
    last_match_date
FROM {CATALOG}.silver.dim_team
""")

# --- dim_venue: resolve canonical venue (falls back to raw) ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_venue AS
SELECT
    venue_key,
    venue_name,
    COALESCE(canonical_venue_name, venue_name) AS display_venue,
    canonical_venue_key,
    city,
    first_match_date,
    last_match_date
FROM {CATALOG}.silver.dim_venue
""")

In [0]:
# --- dim_series / dim_calendar / dim_phase: near pass-through ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_series AS
SELECT series_key, event_name, season, season_start_year,
       match_format, first_match_date, last_match_date
FROM {CATALOG}.silver.dim_series
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_calendar AS
SELECT date_key, date, year, month, day,
       month_name, day_name, quarter, iso_week, is_weekend
FROM {CATALOG}.silver.dim_calendar
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_phase AS
SELECT phase_key, phase_name, is_powerplay, powerplay_type
FROM {CATALOG}.silver.dim_phase
""")

In [0]:
# --- dim_series / dim_calendar / dim_phase: near pass-through ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_series AS
SELECT series_key, event_name, season, season_start_year,
       match_format, first_match_date, last_match_date
FROM {CATALOG}.silver.dim_series
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_calendar AS
SELECT date_key, date, year, month, day,
       month_name, day_name, quarter, iso_week, is_weekend
FROM {CATALOG}.silver.dim_calendar
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_phase AS
SELECT phase_key, phase_name, is_powerplay, powerplay_type
FROM {CATALOG}.silver.dim_phase
""")

In [0]:
# --- dim_match: gold view, with the same FK hashes the fact uses ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_match AS
SELECT
    match_id,
    match_type, match_format, competition_variant, is_international,
    gender, season, season_start_year,
    event_name, event_match_number, event_group, event_stage,
    team_a, team_b, venue, city,
    start_date, end_date,
    CAST(date_format(start_date, 'yyyyMMdd') AS INT)            AS date_key,
    xxhash64(concat_ws('|', coalesce(event_name,'(no event)'), season)) AS series_key,
    xxhash64(lower(trim(venue)))                               AS venue_key,
    balls_per_over, scheduled_overs,
    toss_winner, toss_decision, toss_uncontested,
    outcome_winner, outcome_result, won_by_team,
    outcome_by_runs, outcome_by_wickets, outcome_by_innings,
    outcome_method, outcome_eliminator, outcome_bowl_out
FROM {CATALOG}.silver.dim_match
""")

In [0]:
# ─────────────────────────────────────────────────────────────
# Verify the star resolves entirely within gold now
# ─────────────────────────────────────────────────────────────
spark.sql(f"""
  SELECT t.display_team AS bowling_team,
         v.display_venue AS venue,
         count(*) balls
  FROM {CATALOG}.gold.fact_ball f
  JOIN {CATALOG}.gold.dim_team  t ON f.bowling_team_key = t.team_key
  JOIN {CATALOG}.gold.dim_venue v ON f.venue_key       = v.venue_key
  JOIN {CATALOG}.gold.dim_phase p ON f.phase_key       = p.phase_key
  GROUP BY ALL ORDER BY balls DESC LIMIT 5
""").show(truncate=False)

# every gold FK resolves (no orphans across the whole star)
for dim, key in [("dim_team","batting_team_key"),("dim_venue","venue_key"),
                 ("dim_series","series_key"),("dim_calendar","date_key"),
                 ("dim_phase","phase_key"),("dim_player","batter_id")]:
    pk = "person_id" if dim=="dim_player" else key.replace("batting_team_key","team_key").replace("bowling_team_key","team_key")
    n = spark.sql(f"""
      SELECT count(*) c FROM {CATALOG}.gold.fact_ball f
      LEFT JOIN {CATALOG}.gold.{dim} d ON f.{key} = d.{pk}
      WHERE d.{pk} IS NULL AND f.{key} IS NOT NULL
    """).collect()[0]["c"]
    print(f"orphan {key} -> {dim}: {n}")

In [0]:
# how many balls total?
print("fact rows:", spark.table(f"{CATALOG}.gold.fact_ball").count())

# join team only
print("+team:", spark.sql(f"""
  SELECT count(*) FROM {CATALOG}.gold.fact_ball f
  JOIN {CATALOG}.gold.dim_team t ON f.bowling_team_key = t.team_key
""").collect()[0][0])

# +venue
print("+venue:", spark.sql(f"""
  SELECT count(*) FROM {CATALOG}.gold.fact_ball f
  JOIN {CATALOG}.gold.dim_team  t ON f.bowling_team_key = t.team_key
  JOIN {CATALOG}.gold.dim_venue v ON f.venue_key = v.venue_key
""").collect()[0][0])

# +phase
print("+phase:", spark.sql(f"""
  SELECT count(*) FROM {CATALOG}.gold.fact_ball f
  JOIN {CATALOG}.gold.dim_team  t ON f.bowling_team_key = t.team_key
  JOIN {CATALOG}.gold.dim_venue v ON f.venue_key = v.venue_key
  JOIN {CATALOG}.gold.dim_phase p ON f.phase_key = p.phase_key
""").collect()[0][0])

In [0]:
spark.sql(f"""
  SELECT
    count(*) total,
    count(bowling_team_key) has_bowling_key,
    count(venue_key) has_venue_key,
    count(phase_key) has_phase_key
  FROM {CATALOG}.gold.fact_ball
""").show()

In [0]:
# what does the fact hold vs what the dim holds?
spark.sql(f"""
  SELECT bowling_team_key FROM {CATALOG}.gold.fact_ball LIMIT 5
""").show()

spark.sql(f"""
  SELECT team_key, team_name FROM {CATALOG}.gold.dim_team LIMIT 5
""").show()

In [0]:
spark.sql(f"""
  SELECT count(DISTINCT bowling_team_key) teams,
         count(DISTINCT venue_key) venues,
         count(DISTINCT series_key) series,
         count(DISTINCT date_key) dates
  FROM {CATALOG}.gold.fact_ball
""").show()

In [0]:
spark.sql(f"""
  SELECT count(DISTINCT bowling_team_key) teams,
         count(DISTINCT venue_key) venues,
         count(DISTINCT series_key) series,
         count(DISTINCT date_key) dates
  FROM {CATALOG}.gold.fact_ball
""").show()

In [0]:
from pyspark.sql import functions as F
CATALOG = "cricket"

d  = spark.table(f"{CATALOG}.silver.deliveries")
bp = spark.table(f"{CATALOG}.silver.ball_phase")
dm = spark.table(f"{CATALOG}.silver.dim_match").select(
        "match_id","start_date","event_name","season","venue")

mk = (dm
    .withColumn("date_key",   F.date_format("start_date","yyyyMMdd").cast("int"))
    .withColumn("series_key", F.xxhash64(F.concat_ws("|",
                    F.coalesce("event_name", F.lit("(no event)")), "season")))
    .withColumn("venue_key",  F.xxhash64(F.lower(F.trim(F.col("venue")))))
    .select("match_id","date_key","series_key","venue_key"))

fact = (d
    .join(bp, ["match_id","innings_number","over_number","ball_seq"], "left")
    .join(mk, "match_id", "left")
    .withColumn("batting_team_key", F.xxhash64(F.lower(F.trim(F.col("batting_team")))))
    .withColumn("bowling_team_key", F.xxhash64(F.lower(F.trim(F.col("bowling_team")))))
    .withColumn("is_legal_ball",
        (F.col("extra_wides")==0) & (F.col("extra_noballs")==0))
    .withColumn("is_bowler_wicket",
        F.col("is_wicket") &
        F.col("dismissal_kind").isin("bowled","caught","caught and bowled",
                                     "lbw","stumped","hit wicket")))

fact_ball = fact.select(
    "match_id","innings_number","over_number","ball_seq","is_super_over",
    "date_key","series_key","venue_key","batting_team_key","bowling_team_key",
    "batter_id","bowler_id","non_striker_id","player_out_id","phase_key",
    "runs_batter","runs_extras","runs_total",
    "extra_wides","extra_noballs","extra_byes","extra_legbyes","extra_penalty",
    "is_legal_ball","is_wicket","is_bowler_wicket","wicket_count","non_boundary")

fact_ball.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("match_id").saveAsTable(f"{CATALOG}.gold.fact_ball")

In [0]:
print("fact:", spark.table(f"{CATALOG}.gold.fact_ball").count())
print("+team:", spark.sql(f"""
  SELECT count(*) FROM {CATALOG}.gold.fact_ball f
  JOIN {CATALOG}.gold.dim_team t ON f.bowling_team_key = t.team_key""").collect()[0][0])
print("+venue+phase:", spark.sql(f"""
  SELECT count(*) FROM {CATALOG}.gold.fact_ball f
  JOIN {CATALOG}.gold.dim_team  t ON f.bowling_team_key = t.team_key
  JOIN {CATALOG}.gold.dim_venue v ON f.venue_key = v.venue_key
  JOIN {CATALOG}.gold.dim_phase p ON f.phase_key = p.phase_key""").collect()[0][0])

# and the real cross-dim slice that returned empty before
spark.sql(f"""
  SELECT t.display_team AS bowling_team, count(*) balls
  FROM {CATALOG}.gold.fact_ball f
  JOIN {CATALOG}.gold.dim_team t ON f.bowling_team_key = t.team_key
  GROUP BY ALL ORDER BY balls DESC LIMIT 5
""").show(truncate=False)

In [0]:
CATALOG = "cricket"

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.batting_innings AS
WITH bat AS (
  -- batting contribution per player-innings (from balls faced)
  SELECT
    f.batter_id                        AS person_id,
    f.match_id, f.innings_number,
    sum(f.runs_batter)                                              AS runs,
    sum(CASE WHEN f.is_ball_faced THEN 1 ELSE 0 END)               AS balls,
    sum(CASE WHEN f.runs_batter=4 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS fours,
    sum(CASE WHEN f.runs_batter=6 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS sixes
  FROM {CATALOG}.gold.fact_ball f
  WHERE NOT f.is_super_over            -- convention: super-overs excluded
  GROUP BY f.batter_id, f.match_id, f.innings_number
),
dism AS (
  -- dismissals that count (retired hurt/not out are NOT dismissals)
  SELECT DISTINCT
    w.player_out_id                    AS person_id,
    w.match_id, w.innings_number
  FROM {CATALOG}.silver.wicket w
  JOIN {CATALOG}.silver.dim_innings i
    ON w.match_id = i.match_id AND w.innings_number = i.innings_number
  WHERE NOT i.is_super_over
    AND w.kind NOT IN ('retired hurt','retired not out')
),
-- an innings exists if he batted a ball OR was dismissed
innings_keys AS (
  SELECT person_id, match_id, innings_number FROM bat
  UNION
  SELECT person_id, match_id, innings_number FROM dism
)
SELECT
  ik.person_id,
  ik.match_id,
  ik.innings_number,
  m.season,
  m.event_name,
  m.match_format,
  m.is_international,
  m.start_date,
  COALESCE(b.runs, 0)                  AS runs,
  COALESCE(b.balls, 0)                 AS balls_faced,
  COALESCE(b.fours, 0)                 AS fours,
  COALESCE(b.sixes, 0)                 AS sixes,
  (d.person_id IS NOT NULL)            AS was_out,
  (d.person_id IS NULL)                AS not_out
FROM innings_keys ik
LEFT JOIN bat  b ON ik.person_id=b.person_id AND ik.match_id=b.match_id AND ik.innings_number=b.innings_number
LEFT JOIN dism d ON ik.person_id=d.person_id AND ik.match_id=d.match_id AND ik.innings_number=d.innings_number
JOIN {CATALOG}.gold.dim_match m ON ik.match_id = m.match_id
""")

print("created batting_innings view")

In [0]:
#spark.sql(f"""CREATE TABLE IF NOT EXISTS {CATALOG}.silver.dim_player_enrichment (
#    person_id       STRING,
#    full_name       STRING,
#    country         STRING,
#    dob             DATE,
#    batting_style   STRING,
#    bowling_style   STRING,
#    bat_role        STRING,
#    bowl_role       STRING,
#    is_wicketkeeper BOOLEAN,
#    source          STRING,
#    enriched_at     TIMESTAMP
#)""")